# TP1

## Imports

In [6]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
from urllib.parse import urljoin

##  Télécharger automatiquement
les 51fichiers PDF listés sur https://max.de.wilde.web.ulb.be/camille/.



In [7]:
eaders = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

root_url = "https://max.de.wilde.web.ulb.be/camille/"
response = requests.get(root_url, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

pdf_links = []
seen_urls = set()

# Stratégie principale : tout lien <a href="...pdf">
for link in soup.find_all("a", href=True):
    href = link["href"]
    if href.lower().split("?")[0].endswith(".pdf"):
        pdf_url = urljoin(root_url, href)
        if pdf_url not in seen_urls:
            seen_urls.add(pdf_url)
            title = link.get_text(strip=True) or os.path.basename(pdf_url)
            pdf_links.append([pdf_url, title])

# Repli : si rien n'a été trouvé via les balises <a>, on cherche directement
# des chemins ".pdf" dans le code source brut de la page
if not pdf_links:
    for href in re.findall(r'href=["\']([^"\']+\.pdf)["\']', response.text, flags=re.IGNORECASE):
        pdf_url = urljoin(root_url, href)
        if pdf_url not in seen_urls:
            seen_urls.add(pdf_url)
            pdf_links.append([pdf_url, os.path.basename(pdf_url)])
    print("Repli par expression régulière utilisé.")

print(f"{len(pdf_links)} fichiers PDF trouvés.")

51 fichiers PDF trouvés.


In [4]:
# Affichage du nombre d'articles récupérés
len(articles)

0

In [8]:
# Affichage des liens trouvés
pdf_links[:51]

[['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
  'KB_JB230_1892-08-07_01-0003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
  'KB_JB427_1920-01-10_01-00004.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
  'KB_JB555_1836-02-08_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
  'KB_JB638_1860-05-21_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
  'KB_JB773_1918-11-30_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
  'KB_JB838_1887-12-28_01-00003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
  'KB_JB230_1903-10-16_01-0002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
  'KB_JB427_1933-01-04_01-00003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00

## Création d'un dataframe avec les liens et les titres des articles


In [9]:
df = pd.DataFrame(pdf_links, columns=["url", "titre"])
df["nom_fichier"] = df["url"].apply(lambda u: os.path.basename(u))
df.head()

,url,titre,nom_fichier
0,https://max.de.wilde.web.ulb.be/camille/KB_JB2...,KB_JB230_1892-08-07_01-0003.pdf,KB_JB230_1892-08-07_01-0003.pdf
1,https://max.de.wilde.web.ulb.be/camille/KB_JB4...,KB_JB427_1920-01-10_01-00004.pdf,KB_JB427_1920-01-10_01-00004.pdf
2,https://max.de.wilde.web.ulb.be/camille/KB_JB5...,KB_JB555_1836-02-08_01-00002.pdf,KB_JB555_1836-02-08_01-00002.pdf
3,https://max.de.wilde.web.ulb.be/camille/KB_JB6...,KB_JB638_1860-05-21_01-00002.pdf,KB_JB638_1860-05-21_01-00002.pdf
4,https://max.de.wilde.web.ulb.be/camille/KB_JB7...,KB_JB773_1918-11-30_01-00002.pdf,KB_JB773_1918-11-30_01-00002.pdf


In [ ]:
# Sauvegarde du dataframe dans un fichier csv
df.to_csv(f"../data/rtbf_{time.strftime('%Y%m%d')}.csv", index=False)

## Sauvegarde de l'inventaire des fichiers PDF dans un CSV

In [10]:
def sauvegarder_dataframe(df, dossier="../data", prefixe="camille_pdfs"):
    """
    Sauvegarde un dataframe en CSV avec horodatage,
    en créant le dossier de destination si nécessaire.
    """
    try:
        Path(dossier).mkdir(parents=True, exist_ok=True)
        nom_fichier = f"{prefixe}_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        chemin = Path(dossier) / nom_fichier
        df.to_csv(chemin, index=False, encoding="utf-8-sig")
        print(f"✅ Fichier sauvegardé : {chemin}")
        print(f"   {len(df)} lignes, {len(df.columns)} colonnes")
        return chemin
    except PermissionError:
        print(f"Erreur : permission refusée pour écrire dans '{dossier}'")
    except Exception as e:
        print(f"Erreur lors de la sauvegarde : {e}")

chemin_csv = sauvegarder_dataframe(df)

✅ Fichier sauvegardé : ..\data\camille_pdfs_20260816_174810.csv
   51 lignes, 3 colonnes


## Téléchargement automatique des fichiers PDF

In [11]:
def telecharger_pdf(url, chemin_destination, headers=headers, max_tentatives=3, delai_entre_tentatives=2):
    """
    Télécharge un fichier PDF depuis `url` vers `chemin_destination`.
    Réessaie en cas d'échec (backoff simple), et écrit en flux pour
    ne pas charger tout le fichier en mémoire.
    """
    for tentative in range(1, max_tentatives + 1):
        try:
            with requests.get(url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(chemin_destination, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            return True
        except requests.exceptions.RequestException as e:
            print(f"   ⚠️ Tentative {tentative}/{max_tentatives} échouée pour {url} : {e}")
            if tentative < max_tentatives:
                time.sleep(delai_entre_tentatives * tentative)
    return False


dossier_pdfs = Path("../data/pdfs")
dossier_pdfs.mkdir(parents=True, exist_ok=True)

reussis = []
echoues = []

for i, row in df.iterrows():
    url = row["url"]
    nom_fichier = row["nom_fichier"]
    chemin_destination = dossier_pdfs / nom_fichier

    if chemin_destination.exists():
        print(f"[{i + 1}/{len(df)}] Déjà téléchargé : {nom_fichier}")
        reussis.append(nom_fichier)
        continue

    print(f"[{i + 1}/{len(df)}] Téléchargement de {nom_fichier} ...")
    succes = telecharger_pdf(url, chemin_destination)

    if succes:
        reussis.append(nom_fichier)
    else:
        echoues.append(nom_fichier)

    # Petite pause pour rester correct vis-à-vis du serveur
    time.sleep(0.5)

print("\n=====================================")
print(f"{len(reussis)} fichiers téléchargés avec succès")
print(f"{len(echoues)} échecs")
if echoues:
    print("Fichiers en échec :", echoues)


[1/51] Téléchargement de KB_JB230_1892-08-07_01-0003.pdf ...
[2/51] Téléchargement de KB_JB427_1920-01-10_01-00004.pdf ...
[3/51] Téléchargement de KB_JB555_1836-02-08_01-00002.pdf ...
[4/51] Téléchargement de KB_JB638_1860-05-21_01-00002.pdf ...
[5/51] Téléchargement de KB_JB773_1918-11-30_01-00002.pdf ...
[6/51] Téléchargement de KB_JB838_1887-12-28_01-00003.pdf ...
[7/51] Téléchargement de KB_JB230_1903-10-16_01-0002.pdf ...
[8/51] Téléchargement de KB_JB427_1933-01-04_01-00003.pdf ...
[9/51] Téléchargement de KB_JB555_1899-01-19_01-00003.pdf ...
[10/51] Téléchargement de KB_JB638_1902-12-20_01-00002.pdf ...
[11/51] Téléchargement de KB_JB773_1933-10-07_01-00007.pdf ...
[12/51] Téléchargement de KB_JB838_1911-08-03_01-00006.pdf ...
[13/51] Téléchargement de KB_JB230_1913-07-05_01-0001.pdf ...
[14/51] Téléchargement de KB_JB427_1949-07-18_01-00008.pdf ...
[15/51] Téléchargement de KB_JB555_1940-03-01_01-00004.pdf ...
[16/51] Téléchargement de KB_JB638_1946-07-18_01-00003.pdf ...
[17/

## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp
- Documentation `requests` (téléchargement en flux) : https://requests.readthedocs.io/en/latest/user/quickstart/#raw-response-content